# Boltz-2 — VAM Stage 0 Reference Notebook (PAG-1 / Scenario A)

**VAM Version:** V1.6.1  |  **Scenario:** A (short peptide ≤ 30 aa) + VH/VL  |  **License:** Boltz-2 MIT

This notebook is the canonical Stage 0 deployment for VAM V1.6.1 short-peptide projects.
It implements:

- Boltz-2 weights cached on Google Drive (no re-download per session)
- Empty MSA for both antibody and short-peptide chains
- Soft contact constraint between ECD residue and CDR-H3 centre
- Built-in affinity head enabled (`properties.affinity.binder: B`)
- Dual acceptance gate: ipTM ≥ 0.80 AND PAE_interface < 5.0 Å

**Forbidden tools (V1.6.1):** AF3 server, Chai-1 without license. See `docs/VIRTUAL_AFFINITY_MATURATION_STANDARD_V1.6.md` §2.4.

**Hardware:** Free-tier T4 (16 GB) is sufficient for `< 300` aa total. PAG-1 + Fv ≈ 250 aa. Pro ($10/month) recommended for batch ≥ 50 mutants.

## 1. Mount Drive and prepare cache

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
PROJECT_ROOT = '/content/drive/MyDrive/AbEngineCore_VAM'
BOLTZ_CACHE  = f'{PROJECT_ROOT}/boltz_cache'
RUN_DIR      = f'{PROJECT_ROOT}/runs/pag1'
for d in (BOLTZ_CACHE, RUN_DIR):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

# Symlink ~/.boltz -> Drive cache to persist weights across sessions
home_boltz = pathlib.Path.home() / '.boltz'
if home_boltz.exists() and not home_boltz.is_symlink():
    import shutil; shutil.rmtree(home_boltz)
if not home_boltz.exists():
    os.symlink(BOLTZ_CACHE, str(home_boltz))
print('Boltz weights cache:', home_boltz, '->', BOLTZ_CACHE)

## 2. Install Boltz-2

In [ ]:
!pip install --quiet boltz
!boltz --help | head -30

## 3. Define WT input (PAG-1 short-peptide example)

Replace the placeholder sequences with your project's actual VH / VL and the PAG-1 ECD epitope.

In [ ]:
VH = 'EVQLVESGGGLVQPGGSLRLSCAASGFTFSSYAMSWVRQAPGKGLEWVSAISGSGGSTYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR' \
     'AAYYDFWSGYYTGYAMDY' \
     'WGQGTLVTVSS'
VL = 'DIQMTQSPSSLSASVGDRVTITCRASQSVSSAVAWYQQKPGKAPKLLIYSASSLYSGVPSRFSGSGSGTDFTLTISSLQPEDFATYYCQQYNSYPLT' \
     'FGGGTKVEIK'
LINKER = 'GGGGSGGGGSGGGGSGGGGS'
ANTIBODY_CHAIN = VH + LINKER + VL

# PAG-1 ECD: 8 aa example placeholder + GS linker for the membrane-side direction
PAG1_ECD = 'XXXXXXXX'   # <-- REPLACE with project-specific 8-aa epitope sequence
ANTIGEN_CHAIN = PAG1_ECD + 'GGGGSGGGGSGGGGS'

print('Antibody chain length:', len(ANTIBODY_CHAIN))
print('Antigen  chain length:', len(ANTIGEN_CHAIN))
print('Total residues:', len(ANTIBODY_CHAIN) + len(ANTIGEN_CHAIN))

## 4. Build the Boltz-2 yaml input (V1.6.1 conventions)

- `msa: empty` on all chains (per Scenario A profile)
- Soft contact restraint between ECD mid-residue and CDR-H3 centre
- Affinity head enabled with `binder: B`

In [ ]:
import yaml

# CDR-H3 centre residue index in the antibody chain (project-specific; calibrate via ANARCI)
CDR_H3_CENTER_INDEX_IN_VH = 105   # update for your numbering

yaml_doc = {
    'version': 1,
    'sequences': [
        {'protein': {'id': 'A', 'sequence': ANTIBODY_CHAIN, 'msa': 'empty'}},
        {'protein': {'id': 'B', 'sequence': ANTIGEN_CHAIN, 'msa': 'empty'}},
    ],
    'constraints': [
        {'contact': {'token1': ['B', 4],
                     'token2': ['A', CDR_H3_CENTER_INDEX_IN_VH],
                     'max_distance': 8.0}},
    ],
    'properties': [
        {'affinity': {'binder': 'B'}},
    ],
}

yaml_path = f'{RUN_DIR}/wt_input.yaml'
with open(yaml_path, 'w') as fh:
    yaml.safe_dump(yaml_doc, fh, sort_keys=False)
print('YAML written to', yaml_path)
!cat $yaml_path

## 5. Run Boltz-2 prediction (WT)

In [ ]:
out_dir = f'{RUN_DIR}/wt_out'
!boltz predict {yaml_path} --out_dir {out_dir} --use_msa_server False

## 6. V1.6.1 Acceptance Gate (Scenario A — short peptide)

Apply the dual gate: **ipTM ≥ 0.80 AND PAE_interface < 5.0 Å**.
If either fails, the model must not enter Stage 1+. Re-run with different seed or escalate to HADDOCK3 with sampling = 200.

In [ ]:
import json, glob, numpy as np, pathlib

IPTM_MIN = 0.80                 # V1.6.1 Scenario A threshold
PAE_INTERFACE_MAX = 5.0          # V1.6.1 Scenario A threshold (Angstroms)

conf_files = glob.glob(f'{out_dir}/**/*confidence*.json', recursive=True)
pae_files  = glob.glob(f'{out_dir}/**/*pae*.npz', recursive=True)
assert conf_files, 'Confidence JSON not found'

with open(conf_files[0]) as fh:
    conf = json.load(fh)
iptm = conf.get('iptm') or conf.get('interface_ptm') or float('nan')
ptm  = conf.get('ptm') or float('nan')
print(f'ipTM = {iptm:.3f}   pTM = {ptm:.3f}')

# PAE interface mean: antigen tokens (chain B residues 0..7 = ECD only, exclude GS linker)
if pae_files:
    pae = np.load(pae_files[0])
    pae_arr = pae[pae.files[0]]
    n_antibody = len(ANTIBODY_CHAIN)
    ag_residue_indices = list(range(n_antibody, n_antibody + 8))   # ECD only, no GS
    ab_cdr_indices = list(range(95, 115))                            # CDR-H3 region (project-calibrate)
    interface_pae = pae_arr[np.ix_(ag_residue_indices, ab_cdr_indices)]
    pae_iface_mean = float(interface_pae.mean())
    print(f'PAE_interface_mean = {pae_iface_mean:.2f} A')
else:
    pae_iface_mean = float('nan')
    print('PAE matrix not found; cannot enforce dual gate')

iptm_ok = iptm >= IPTM_MIN
pae_ok  = pae_iface_mean < PAE_INTERFACE_MAX
verdict = 'PASS' if (iptm_ok and pae_ok) else 'REJECT'
print(f'\nV1.6.1 Acceptance: {verdict}  (ipTM_ok={iptm_ok}, PAE_ok={pae_ok})')
if verdict == 'REJECT':
    print('\n  -> Re-run with different seed or escalate to HADDOCK3 sampling=200 + active-residue restraints')

## 7. Boltz-2 affinity head readout (Stage 7 ensemble third vote)

**V1.6.1 policy:** affinity_pred_value is for **relative ranking among same-WT mutants only**. Never use as absolute IC50/Kd.

In [ ]:
aff_files = glob.glob(f'{out_dir}/**/*affinity*.json', recursive=True)
if aff_files:
    with open(aff_files[0]) as fh:
        aff = json.load(fh)
    print('affinity_pred_value     =', aff.get('affinity_pred_value'))
    print('affinity_probability    =', aff.get('affinity_probability_binary'))
    print('\n  V1.6.1: relative-ranking signal only; do NOT cite as absolute IC50.')
else:
    print('No affinity JSON; check that properties.affinity.binder was set to \'B\' in the yaml.')

## 8. Per-mutation batch runner (Stage 7 ensemble use)

Skeleton for running a list of mutants. Each mutation requires regenerating the yaml with the mutated `ANTIBODY_CHAIN` and re-running Boltz-2. Affinity head outputs feed the Stage 7 ensemble (weight 0.25).

In [ ]:
import copy

def apply_mutation(seq: str, position: int, mut_aa: str) -> str:
    return seq[:position] + mut_aa + seq[position+1:]

MUTATIONS = [
    {'tag': 'W107Y', 'position': 105, 'wt': 'W', 'mut': 'Y'},   # example, project-specific
    {'tag': 'D108N', 'position': 106, 'wt': 'D', 'mut': 'N'},
]

results = []
for m in MUTATIONS:
    mut_seq = apply_mutation(ANTIBODY_CHAIN, m['position'], m['mut'])
    yml = copy.deepcopy(yaml_doc)
    yml['sequences'][0]['protein']['sequence'] = mut_seq
    mut_yaml = f'{RUN_DIR}/{m["tag"]}_input.yaml'
    mut_out  = f'{RUN_DIR}/{m["tag"]}_out'
    with open(mut_yaml, 'w') as fh:
        yaml.safe_dump(yml, fh, sort_keys=False)
    print(f'\n[Boltz-2] running {m["tag"]} ...')
    !boltz predict {mut_yaml} --out_dir {mut_out} --use_msa_server False
    aff_path = glob.glob(f'{mut_out}/**/*affinity*.json', recursive=True)
    if aff_path:
        with open(aff_path[0]) as fh:
            results.append({'tag': m['tag'], **json.load(fh)})

import pandas as pd
df = pd.DataFrame(results)
df.to_csv(f'{RUN_DIR}/affinity_ensemble_vote.csv', index=False)
df

## 9. Hand-off to local AbEngineCore pipeline

Copy the WT PDB and the affinity ensemble CSV to the project root, then run the V1.6 pipeline locally:

```bash
# On local machine
cp /path/to/Drive/runs/pag1/wt_out/*.pdb projects/PAG1/wt_complex.pdb
cp /path/to/Drive/runs/pag1/affinity_ensemble_vote.csv projects/PAG1/boltz2_affinity.csv

conda activate affmat
python scripts/affinity_energy_cli.py \
    --pdb projects/PAG1/wt_complex.pdb \
    --ab-chains A_VH A_VL \
    --ag-chains B \
    --scenario A \
    --tools all
```

The downstream V1.6 pipeline (CHECK 6 design prior + CHECK 7 sequence liability + Stage 2.5 structural Veto + CHECK 8 relax/clash gate) runs on local CPU; only Stage 8/10 MD requires GPU and may again use Colab.